# UNMIX for iPad — dialogue / music splitter

Same separation engine as the desktop `UNMIXv1.0.py` (Meta's **Demucs v4**), but the
heavy lifting happens on Google's machines, so an iPad is all you need.

**Before you start — this takes 30 seconds and makes it ~10x faster:**

`Runtime` -> `Change runtime type` -> **T4 GPU** -> `Save`

Then run the cells below in order. Tap the round play button on the left of each one.


## 1. Install the engine

First run takes 2-3 minutes. You only do this once per session.


In [ ]:
#@title Install Demucs { display-mode: "form" }
import subprocess, sys, textwrap

print("Installing Demucs... this takes a couple of minutes.\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "demucs==4.0.1", "soundfile"], check=True)

import torch
dev = "GPU (fast)" if torch.cuda.is_available() else "CPU (slow - see note below)"
print("Ready. Running on:", dev)
if not torch.cuda.is_available():
    print(textwrap.dedent("""
        No GPU attached. It will still work, just 5-10x slower.
        To switch: Runtime -> Change runtime type -> T4 GPU -> Save,
        then run this cell again.
    """))

## 2. Load your file

Tap **Choose Files**, then pick your CDenza export from the Files app or from Photos.

Audio (`.wav .mp3 .m4a .aac .flac`) or video (`.mp4 .mov`) both work — if you give it
a video, the audio track is pulled out automatically and the video is left alone.


In [ ]:
#@title Upload { display-mode: "form" }
import os, shutil, subprocess
from pathlib import Path
from google.colab import files

WORK = Path("/content/unmix"); shutil.rmtree(WORK, ignore_errors=True)
(WORK / "in").mkdir(parents=True, exist_ok=True)

up = files.upload()
if not up:
    raise SystemExit("Nothing uploaded. Run this cell again and pick a file.")

name = list(up.keys())[0]
raw = WORK / "in" / name
raw.write_bytes(up[name])

VIDEO = {".mp4", ".mov", ".mkv", ".webm", ".avi", ".m4v"}
if raw.suffix.lower() in VIDEO:
    print("\nVideo file - extracting the audio track...")
    SOURCE = WORK / "in" / (raw.stem + "_audio.wav")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(raw),
                    "-vn", "-acodec", "pcm_s24le", str(SOURCE)], check=True)
else:
    SOURCE = raw

probe = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration",
     "-of", "default=nw=1:nk=1", str(SOURCE)],
    capture_output=True, text=True)
try:
    secs = float(probe.stdout.strip())
    print(f"\nLoaded: {SOURCE.name}  ({int(secs//60)}m {int(secs%60)}s)")
except ValueError:
    print(f"\nLoaded: {SOURCE.name}")

## 3. Split it

`Standard` is the right choice almost every time. If the split sounds smeared —
music bleeding into the dialogue, or a word swallowed — run it again on
`Best quality`, which is the same model fine-tuned and takes about 4x longer.

`passes` trades time for cleanliness. Leave it at 1 unless you have a stubborn cue.


In [ ]:
#@title Separate { display-mode: "form" }
model = "htdemucs" #@param ["htdemucs", "htdemucs_ft", "mdx_extra"]
passes = 1 #@param {type:"slider", min:1, max:4, step:1}

import subprocess, sys, time
from pathlib import Path

OUT = WORK / "out"
start = time.time()

cmd = [sys.executable, "-m", "demucs",
       "-n", model,
       "--two-stems", "vocals",       # dialogue vs everything else
       "--shifts", str(passes),
       "-o", str(OUT),
       str(SOURCE)]

print("Separating. Progress bar below - GPU runs roughly 10x faster than real time.\n")
subprocess.run(cmd, check=True)

stem_dir = next((OUT / model).glob("*"))
dialogue = stem_dir / "vocals.wav"
music    = stem_dir / "no_vocals.wav"

final = WORK / "stems"; final.mkdir(exist_ok=True)
base = Path(SOURCE).stem.replace("_audio", "")
dialogue.rename(final / f"{base}_DIALOGUE.wav")
music.rename(final / f"{base}_MUSIC_FX.wav")

print(f"\nDone in {int(time.time()-start)}s.")
for f in sorted(final.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size/1e6:.1f} MB)")

## 4. Listen before you commit

Check the split here rather than after you've rebuilt your timeline.


In [ ]:
#@title Preview both stems { display-mode: "form" }
import IPython.display as ipd
from pathlib import Path

for f in sorted(Path(WORK / "stems").iterdir()):
    label = "DIALOGUE" if "DIALOGUE" in f.name else "MUSIC + FX"
    print(label)
    ipd.display(ipd.Audio(str(f)))

## 5. Send them to your iPad

Safari will ask once per file. Choose **Download** — they land in
`Files -> Downloads`, ready to drag into LumaFusion, iMovie, or Final Cut.

If Safari blocks the second download, tap the `aA` icon in the address bar ->
`Website Settings` -> allow downloads, then run this cell again.


In [ ]:
#@title Download { display-mode: "form" }
from google.colab import files
from pathlib import Path
import time

for f in sorted(Path(WORK / "stems").iterdir()):
    files.download(str(f))
    time.sleep(2)   # Safari drops downloads fired back-to-back

---

### If the dialogue stem still has music under it

Demucs was trained on sung vocals, not spoken dialogue, so a quiet line under a
loud score is the hard case. Two things that reliably help:

1. Re-run step 3 with `htdemucs_ft` and `passes = 2`.
2. Cut the scene into shorter pieces and split them separately. The model
   normalises against the whole file, so one loud music-only stretch can drag
   down how it treats the quiet dialogue elsewhere.

### Nothing is kept

The Colab machine is wiped when you close the tab. Re-running the notebook
tomorrow starts from step 1.
